# Baseera — obstacle model (YOLO11n)

Trains the model the app runs on frames from the glasses camera.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Everything worth keeping goes to Google Drive (`MyDrive/baseera`): the
dataset (zipped once, so later sessions skip the download), training
checkpoints (a disconnected session resumes where it stopped) and the
exported model. Run the cells top to bottom; after a disconnect, run them
again — finished steps are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/baseera'
DATA = '/content/obstacles'
RUN = f'{BASE}/runs/obstacles'
EXPORT = f'{BASE}/export'
PHOTOS = f'{BASE}/glasses_photos'   # optional: your labelled glasses photos
for d in (BASE, EXPORT):
    os.makedirs(d, exist_ok=True)

In [ ]:
!pip -q install ultralytics fiftyone

The two scripts from `ml/obstacle` in the Baseera project (kept in sync by `make_notebook.py`).

In [ ]:
%%writefile classes.py
"""Classes of Baseera's obstacle model.

Every label is an ObstacleKind name in the app (app/lib/data/obstacles.dart),
so the app turns a detection straight into a warning ("كرسي على يمينك").
Keep the order: the exported model's class ids follow it, and so does
classes.txt for labelling your own photos.

Stage 1 glasses have only the camera, so this model is the only thing that
sees obstacles: "obstacle" covers things in the way that have no class of
their own (plant pots, sofas, beds, suitcases, fire hydrants). The camera can't
tell stairs going up from down; "stairs" is warned as a fall risk. Drops
(curbs, holes) need the downward distance sensor of the later glasses.
"""

LABELS = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "dog",
    "chair",
    "bench",
    "table",
    "door",
    "stairs",
    "pole",
    "tree",
    "obstacle",
]

# COCO 2017 label -> ours. COCO boxes every one of its 80 classes in every
# image, so its images also teach the model what is NOT one of these.
COCO = {
    "person": "person",
    "bicycle": "bicycle",
    "car": "car",
    "truck": "car",
    "motorcycle": "motorcycle",
    "bus": "bus",
    "dog": "dog",
    "chair": "chair",
    "bench": "bench",
    "dining table": "table",
    # in the way, no class of their own
    "potted plant": "obstacle",
    "fire hydrant": "obstacle",
    "couch": "obstacle",
    "bed": "obstacle",
    "suitcase": "obstacle",
    # signs and meters stand on a pole
    "parking meter": "pole",
    "stop sign": "pole",
}

# Open Images V7 label -> ours, for what COCO lacks.
OPEN_IMAGES_TARGETS = {
    "Door": "door",
    "Stairs": "stairs",
    "Tree": "tree",
    "Street light": "pole",
}

# Also kept when they show up in those Open Images photos, so a person
# standing in a doorway isn't learned as background.
OPEN_IMAGES_ALSO = {
    "Person": "person",
    "Man": "person",
    "Woman": "person",
    "Boy": "person",
    "Girl": "person",
    "Car": "car",
    "Truck": "car",
    "Bicycle": "bicycle",
    "Motorcycle": "motorcycle",
    "Bus": "bus",
    "Dog": "dog",
    "Chair": "chair",
    "Bench": "bench",
    "Table": "table",
}

# Photos to download per source class. Doors and stairs matter most indoors
# and are rarer in public data, so they get more.
COCO_PER_CLASS = 1200
# The "obstacle" and extra "pole" sources need fewer each.
COCO_PER_CLASS_OVERRIDE = {
    "potted plant": 600,
    "fire hydrant": 600,
    "couch": 600,
    "bed": 400,
    "suitcase": 600,
    "parking meter": 500,
    "stop sign": 500,
}
OPEN_IMAGES_PER_CLASS = {
    "Door": 3000,
    "Stairs": 3000,
    "Tree": 1500,
    "Street light": 2000,
}

In [ ]:
%%writefile build_dataset.py
"""Builds Baseera's obstacle dataset in YOLO format.

Sources (downloaded through FiftyOne, no accounts needed):
  * COCO 2017      people, vehicles, dogs, chairs, benches, tables, and
                   general obstacles (plant pots, sofas, hydrants...)
  * Open Images V7 doors, stairs, trees, street-light poles
  * your own labelled glasses-camera photos (--photos), which matter most:
    the glasses camera is low quality and wide-angle, unlike these photos.

    python build_dataset.py --out /content/obstacles
    python build_dataset.py --out /content/obstacles --photos /path/to/photos

--photos is a folder with images/ and labels/ (YOLO .txt files) labelled with
the classes in classes.txt, which this script writes next to the dataset.
Every 10th image (by file name) goes to validation.
"""

import argparse
import os
import shutil
import zlib
from collections import Counter
from pathlib import Path

from classes import (
    COCO,
    COCO_PER_CLASS,
    COCO_PER_CLASS_OVERRIDE,
    LABELS,
    OPEN_IMAGES_ALSO,
    OPEN_IMAGES_PER_CLASS,
    OPEN_IMAGES_TARGETS,
)

SEED = 51
IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def split_of(name: str) -> str:
    return "val" if zlib.crc32(name.encode()) % 10 == 0 else "train"


class Writer:
    def __init__(self, out: Path):
        self.out = out
        self.names: set[str] = set()
        self.boxes = Counter()
        self.images = Counter()
        for split in ("train", "val"):
            (out / "images" / split).mkdir(parents=True, exist_ok=True)
            (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    def add(self, image: str, name: str, lines: list[str]) -> None:
        """One image and its YOLO label lines (skipped if already added)."""
        if name in self.names or not lines:
            return
        self.names.add(name)
        split = split_of(name)
        target = self.out / "images" / split / name
        try:
            os.link(image, target)  # no copy when on the same disk
        except OSError:
            shutil.copy2(image, target)
        (self.out / "labels" / split / Path(name).with_suffix(".txt")).write_text(
            "\n".join(lines) + "\n"
        )
        self.images[split] += 1
        for line in lines:
            self.boxes[LABELS[int(line.split()[0])]] += 1

    def add_fiftyone(self, dataset, mapping: dict[str, str], prefix: str) -> None:
        for sample in dataset.iter_samples(progress=True):
            detections = sample["ground_truth"]
            lines = []
            for d in detections.detections if detections else []:
                label = mapping.get(d.label)
                if label is None:
                    continue
                x, y, w, h = d.bounding_box  # top-left x, y, w, h in 0..1
                lines.append(
                    f"{LABELS.index(label)} {x + w / 2:.6f} {y + h / 2:.6f} {w:.6f} {h:.6f}"
                )
            self.add(sample.filepath, f"{prefix}_{Path(sample.filepath).name}", lines)

    def write_yaml(self) -> Path:
        names = "\n".join(f"  {i}: {label}" for i, label in enumerate(LABELS))
        yaml = self.out / "dataset.yaml"
        yaml.write_text(
            f"path: {self.out.resolve().as_posix()}\n"
            "train: images/train\nval: images/val\n"
            f"names:\n{names}\n"
        )
        (self.out / "classes.txt").write_text("\n".join(LABELS) + "\n")
        return yaml


def add_coco(writer: Writer) -> None:
    import fiftyone.zoo as foz

    for source in COCO:
        dataset = foz.load_zoo_dataset(
            "coco-2017",
            split="train",
            label_types=["detections"],
            classes=[source],
            max_samples=COCO_PER_CLASS_OVERRIDE.get(source, COCO_PER_CLASS),
            shuffle=True,
            seed=SEED,
            label_field="ground_truth",
            dataset_name=f"baseera-coco-{source.replace(' ', '-')}",
            drop_existing_dataset=True,
        )
        writer.add_fiftyone(dataset, COCO, "coco")
        dataset.delete()


def add_open_images(writer: Writer) -> None:
    import fiftyone.utils.openimages as fouo
    import fiftyone.zoo as foz

    available = set(fouo.get_classes(version="v7"))
    missing = [c for c in {**OPEN_IMAGES_TARGETS, **OPEN_IMAGES_ALSO} if c not in available]
    if missing:
        print(f"Not in Open Images V7, skipped: {missing}")
    mapping = {
        k: v for k, v in {**OPEN_IMAGES_ALSO, **OPEN_IMAGES_TARGETS}.items() if k in available
    }
    for source, count in OPEN_IMAGES_PER_CLASS.items():
        if source not in available:
            continue
        dataset = foz.load_zoo_dataset(
            "open-images-v7",
            split="train",
            label_types=["detections"],
            classes=[source],
            max_samples=count,
            shuffle=True,
            seed=SEED,
            label_field="ground_truth",
            dataset_name=f"baseera-oi-{source.replace(' ', '-')}",
            drop_existing_dataset=True,
        )
        writer.add_fiftyone(dataset, mapping, "oi")
        dataset.delete()


def add_photos(writer: Writer, folder: Path) -> None:
    """Your own photos, already in YOLO format with our class order."""
    for image in sorted((folder / "images").rglob("*")):
        if image.suffix.lower() not in IMAGE_TYPES:
            continue
        label = folder / "labels" / image.relative_to(folder / "images").with_suffix(".txt")
        lines = label.read_text().split("\n") if label.exists() else []
        lines = [l.strip() for l in lines if l.strip()]
        bad = [l for l in lines if not 0 <= int(l.split()[0]) < len(LABELS)]
        if bad:
            raise SystemExit(f"{label}: class id outside classes.txt: {bad[0]}")
        writer.add(str(image), f"own_{image.name}", lines)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--photos", type=Path, help="your labelled glasses photos")
    parser.add_argument("--skip-public", action="store_true", help="only --photos")
    args = parser.parse_args()

    writer = Writer(args.out)
    if not args.skip_public:
        add_coco(writer)
        add_open_images(writer)
    if args.photos:
        add_photos(writer, args.photos)
    yaml = writer.write_yaml()

    print(f"\nWrote {yaml}")
    print(f"images: {dict(writer.images)}")
    for label in LABELS:
        print(f"  {label:<10} {writer.boxes[label]:>6} boxes")


if __name__ == "__main__":
    main()

## 1. Quick start (optional): stock COCO model

Exports the off-the-shelf YOLO11n (people, vehicles, chairs, tables…) so the
app can be tested with real glasses before training finishes: install it from
`MyDrive/baseera/export/coco`. Doors, stairs, poles and trees only come with
the trained model.

In [ ]:
from ultralytics import YOLO
import glob, shutil

coco = YOLO('yolo11n.pt')
# format='saved_model' writes the TFLite files the app reads (NHWC input).
# Don't use format='tflite': Ultralytics >= 8.4.83 turns it into a LiteRT
# export with a different file name and layout.
coco.export(format='saved_model', imgsz=320)
f16 = glob.glob('yolo11n_saved_model/*float16.tflite')[0]
os.makedirs(f'{EXPORT}/coco', exist_ok=True)
shutil.copy(f16, f'{EXPORT}/coco/obstacles.tflite')
with open(f'{EXPORT}/coco/obstacle_labels.txt', 'w') as f:
    f.write('\n'.join(coco.names[i] for i in range(len(coco.names))))
print('Saved to', f'{EXPORT}/coco')

## 2. Dataset

About 24k photos from COCO and Open Images (first run: ~40–50 min, mostly
downloading). Your own glasses photos are added from `MyDrive/baseera/glasses_photos`
if that folder exists (YOLO format, classes in the order of `classes.txt`).

In [ ]:
ZIP = f'{BASE}/obstacles_dataset.zip'
if os.path.exists(f'{DATA}/dataset.yaml'):
    print('Dataset already here.')
elif os.path.exists(ZIP):
    !unzip -q -o "{ZIP}" -d /content
else:
    photos = f'--photos "{PHOTOS}"' if os.path.isdir(PHOTOS) else ''
    !python build_dataset.py --out {DATA} {photos}
    !cd /content && zip -q -r "{ZIP}" obstacles
print(open(f'{DATA}/dataset.yaml').read())

## 3. Train

320×320 input, matching the glasses' 320×240 frames. ~2–3 min per epoch on a
T4. Checkpoints go to Drive; if the session drops, rerun from the top and
this cell resumes.

In [ ]:
from ultralytics import YOLO

last = f'{RUN}/weights/last.pt'
if os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO('yolo11n.pt').train(
        data=f'{DATA}/dataset.yaml',
        imgsz=320,
        epochs=80,
        batch=64,
        patience=20,
        cos_lr=True,
        close_mosaic=10,
        project=f'{BASE}/runs',
        name='obstacles',
        exist_ok=True,
        workers=4,
    )

## 4. How good is it?

mAP50 per class on the validation photos (higher is better; above ~0.6 is solid for this use).

In [ ]:
best = YOLO(f'{RUN}/weights/best.pt')
metrics = best.val(data=f'{DATA}/dataset.yaml', imgsz=320)
for i, name in best.names.items():
    print(f'{name:<10} mAP50 {metrics.box.ap50[i]:.2f}')
print(f'all        mAP50 {metrics.box.map50:.2f}')

## 5. Export for the app

Then download `MyDrive/baseera/export` and run `python ml/obstacle/install_model.py <that folder>`.

In [ ]:
import glob, shutil

best = YOLO(f'{RUN}/weights/best.pt')
best.export(format='saved_model', imgsz=320)  # see step 1 about 'tflite'
f16 = glob.glob(f'{RUN}/weights/best_saved_model/*float16.tflite')[0]
shutil.copy(f16, f'{EXPORT}/obstacles.tflite')
with open(f'{EXPORT}/obstacle_labels.txt', 'w') as f:
    f.write('\n'.join(best.names[i] for i in range(len(best.names))))
print('Saved to', EXPORT, os.listdir(EXPORT))

## 6. Look at a few predictions

In [ ]:
import random
from IPython.display import Image, display

images = glob.glob(f'{DATA}/images/val/*')
for result in best.predict(random.sample(images, 6), imgsz=320, conf=0.45, verbose=False):
    path = f'/content/pred_{os.path.basename(result.path)}.jpg'
    result.save(filename=path)
    display(Image(path, width=480))